# Maia Corpus 0.3.0 — PT / EN / ES + Logical Reasoning

**Colab corpus-construction notebook · v0.3.0, first reproducible release**

Builds a **licensed, provenance-aware, deduplicated corpus** for the Maia Lite 0.3.0 Chat pretraining notebook. Does **not** reuse model weights or perform training. No large external corpus is silently downloaded. Real texts must be supplied from sources whose rights and usage terms you have reviewed.

**Outputs:** `corpus/source/<lang>/<domain>/*.jsonl` for pretraining; `corpus/evaluation/*.jsonl` for independent tests; `corpus/reports/*.json` for auditing. The pretraining notebook reads `source` recursively and expects `text`, `lang`, `domain`.

**Research design:** language 30%, dialogue 20%, logic 25%, educational 15%, creative 10%; PT 40%, EN 35%, ES 25% are **targets, not automatically achieved**. This notebook reports observed proportions, and offers optional explicit, reproducible sampling. A generator of verified logical arguments is included, but synthetic logic **does not replace** authentic licensed language data.

## 1. Install and initialize directories

In [ ]:
%pip -q install "numpy>=1.26" "pandas>=2.0" "tqdm>=4.66"

In [ ]:
from google.colab import drive
from pathlib import Path
import os, re, json, hashlib, random, unicodedata, collections, math, shutil, csv
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
INBOX = ROOT/'corpus'/'inbox'         # Your curated files: inbox/<lang>/<domain>/...
SOURCE = ROOT/'corpus'/'source'       # Consumed by the pretraining notebook
EVAL = ROOT/'corpus'/'evaluation'     # NEVER put these files under source
REPORTS = ROOT/'corpus'/'reports'
for d in (INBOX, SOURCE, EVAL, REPORTS): d.mkdir(parents=True, exist_ok=True)
SEED=20261018
LANGS=('pt','en','es')
DOMAINS=('language','dialogue','logic','educational','creative')
TARGET_LANG={'pt':.40,'en':.35,'es':.25}
TARGET_DOMAIN={'language':.30,'dialogue':.20,'logic':.25,'educational':.15,'creative':.10}
print('Inbox:', INBOX, '\nPretraining source:', SOURCE, '\nHeld-out:', EVAL)

## 2. Source registry and license policy

Place `.txt` or `.jsonl` files under `inbox/<lang>/<domain>/`. `.txt` is treated as **one document per file**, not one line per document. `.jsonl` must contain a `text` field per record. Register every collection with a stable `source_id`, its license, attribution and provenance. **Unregistered files are rejected**. No web scraping or unlicensed collection is performed. Verify the exact license/terms yourself, including attribution, redistribution, personal data and ML training rights. A permissive license string is not a legal determination.

To add a source, edit `SOURCE_REGISTRY` below. File names can be matched with a prefix via `path_prefix`. Keep generated synthetic logic separate: its provenance is automatically labeled.

In [ ]:
# EXAMPLE ONLY. Edit this registry for your actual input collections.
# path_prefix is relative to INBOX, e.g. 'pt/language/my_collection/'
SOURCE_REGISTRY = [
    # {'path_prefix':'pt/language/my_collection/', 'source_id':'collection-pt-001',
    #  'license':'CC-BY-4.0', 'attribution':'Author, work, URL',
    #  'origin_url':'https://example.org/dataset', 'approved':True},
]

# Deliberate explicit opt-in; leave False until rights have been reviewed.
REQUIRE_APPROVED_SOURCES=True
MIN_CHARS=80
MAX_CHARS=100000
MAX_RECORDS_PER_FILE=None  # Optional bounded pilot, e.g. 5000

def source_info(relative_path):
    key=relative_path.as_posix()
    matches=[r for r in SOURCE_REGISTRY if key.startswith(r['path_prefix'])]
    if len(matches)!=1: raise ValueError(f'Expected one registry entry for {key}, found {len(matches)}')
    entry=matches[0]
    if REQUIRE_APPROVED_SOURCES and not entry.get('approved',False):
        raise ValueError(f'Source not approved: {key}')
    for k in ('source_id','license','attribution','origin_url'):
        if not entry.get(k): raise ValueError(f'Missing {k} for {key}')
    return entry

def clean_text(t):
    t=unicodedata.normalize('NFC',str(t)).replace('\r\n','\n').replace('\r','\n')
    t=re.sub(r'[ \t]+',' ',t)
    t=re.sub(r'\n{4,}','\n\n\n',t)
    return t.strip()

def fingerprint(t):
    norm=unicodedata.normalize('NFKC',t).casefold()
    norm=re.sub(r'\s+',' ',norm).strip()
    return hashlib.sha256(norm.encode('utf-8')).hexdigest()

def iter_registered():
    paths=sorted(p for p in INBOX.rglob('*') if p.is_file() and p.suffix.lower() in ('.txt','.jsonl'))
    for path in paths:
        rel=path.relative_to(INBOX)
        if len(rel.parts)<3 or rel.parts[0] not in LANGS or rel.parts[1] not in DOMAINS:
            raise ValueError(f'Expected inbox/<lang>/<domain>/file: {rel}')
        lang,domain=rel.parts[:2]
        src=source_info(rel)
        if path.suffix.lower()=='.txt':
            items=[{'text':path.read_text(encoding='utf-8')}]
        else:
            items=[]
            with path.open(encoding='utf-8') as f:
                for line_no,line in enumerate(f,1):
                    if line.strip():
                        rec=json.loads(line)
                        if not isinstance(rec,dict) or 'text' not in rec: raise ValueError(f'{rel}:{line_no} missing text')
                        items.append(rec)
                        if MAX_RECORDS_PER_FILE and len(items)>=MAX_RECORDS_PER_FILE: break
        for idx,rec in enumerate(items):
            text=clean_text(rec['text'])
            if not (MIN_CHARS<=len(text)<=MAX_CHARS): continue
            yield {'text':text,'lang':lang,'domain':domain,'source_id':src['source_id'],
                   'license':src['license'],'attribution':src['attribution'],
                   'origin_url':src['origin_url'],'record_id':f'{rel}:{idx}'}
print('Registered collections:',len(SOURCE_REGISTRY))

## 3. Logical reasoning: formally verified synthetic examples

We generate **valid and invalid deductions** with invented terms, using a directed inclusion graph (`All A are B`) and disjointness (`No A is B`). A finite-set model checker verifies whether each candidate conclusion follows from the premises. The checker tests all assignments of membership to the invented predicates for one named individual, which is sufficient for this restricted monadic fragment. It does **not** establish correctness for arbitrary first-order logic.

**Crucial:** train and evaluation use disjoint **structural template families**, not merely different names. We also retain a set of multilingual paired probes outside training. The synthetic examples are a controlled experimental resource, not evidence that the model has learned general reasoning.

In [ ]:
LOGIC_COUNTS_TRAIN_PER_LANG=12000  # pilot size; increase only after auditing
LOGIC_COUNTS_EVAL_PER_LANG=600
# 'train' and 'eval' are structurally distinct families.
TEMPLATES_TRAIN=('chain2_yes','chain3_yes','disjoint_no','affirm_consequent_invalid')
TEMPLATES_EVAL=('chain4_yes','disjoint_chain_no','denying_antecedent_invalid')
NAMES=('Zaf','Luma','Neri','Tavo','Mira','Seli','Vako','Daro','Nima','Ravi')
TERMS=('dax','mipo','luro','feno','zapi','tami','beko','naro','vipo','sanu','kelo','rima')

def sentence(lang,kind,a,b):
    if kind=='all':
        return {'pt':f'Todo {a} é {b}.','en':f'Every {a} is {b}.','es':f'Todo {a} es {b}.'}[lang]
    if kind=='none':
        return {'pt':f'Nenhum {a} é {b}.','en':f'No {a} is {b}.','es':f'Ningún {a} es {b}.'}[lang]
    raise ValueError(kind)

def named(lang,name,term):
    return {'pt':f'{name} é {term}.','en':f'{name} is {term}.','es':f'{name} es {term}.'}[lang]

def query(lang,name,term):
    return {'pt':f'Podemos concluir que {name} é {term}?',
            'en':f'Can we conclude that {name} is {term}?',
            'es':f'¿Podemos concluir que {name} es {term}?'}[lang]

def verify_entailment(rules,facts,conclusion,n):
    # Enumerate all 2**n interpretations of predicates for a single individual.
    # rules: ('all'/'none', i, j), facts: (predicate_index, truth)
    satisfiable=0; counterexamples=0
    for mask in range(1<<n):
        truth=lambda i: bool(mask & (1<<i))
        if any(truth(i)!=value for i,value in facts): continue
        if any((truth(a) and not truth(b)) if op=='all' else (truth(a) and truth(b)) for op,a,b in rules): continue
        satisfiable+=1
        if not truth(conclusion): counterexamples+=1
    if not satisfiable: raise ValueError('Unsatisfiable premises: vacuous entailment')
    return counterexamples==0

def make_logic(lang,template,rng):
    # Randomize labels but hold structure fixed for auditable train/test families.
    term=rng.sample(TERMS,5)
    name=rng.choice(NAMES)
    a,b,c,d,e=term
    if template=='chain2_yes':
        rules=[('all',0,1),('all',1,2)]; facts=[(0,True)]; target=2; n=3
    elif template=='chain3_yes':
        rules=[('all',0,1),('all',1,2),('all',2,3)]; facts=[(0,True)]; target=3; n=4
    elif template=='disjoint_no':
        rules=[('all',0,1),('none',1,2)]; facts=[(0,True)]; target=2; n=3
    elif template=='affirm_consequent_invalid':
        rules=[('all',0,1)]; facts=[(1,True)]; target=0; n=2
    elif template=='chain4_yes':
        rules=[('all',0,1),('all',1,2),('all',2,3),('all',3,4)]; facts=[(0,True)]; target=4; n=5
    elif template=='disjoint_chain_no':
        rules=[('all',0,1),('all',1,2),('none',2,3)]; facts=[(0,True)]; target=3; n=4
    elif template=='denying_antecedent_invalid':
        # A -> B and not A; does not establish B. Negative fact expressed explicitly.
        rules=[('all',0,1)]; facts=[(0,False)]; target=1; n=2
    else: raise ValueError(template)
    result=verify_entailment(rules,facts,target,n)
    # For 'disjoint_no' the positive claim is not entailed, but its negation IS entailed.
    # Label is always whether the queried positive conclusion follows.
    premises=[sentence(lang,op,term[i],term[j]) for op,i,j in rules]
    for i,value in facts:
        if value: premises.append(named(lang,name,term[i]))
        else: premises.append({'pt':f'{name} não é {term[i]}.',
                               'en':f'{name} is not {term[i]}.',
                               'es':f'{name} no es {term[i]}.'}[lang])
    rng.shuffle(premises)
    yes={'pt':'Sim.','en':'Yes.','es':'Sí.'}[lang]
    no={'pt':'Não é possível concluir isso.','en':'That does not follow.','es':'No se puede concluir eso.'}[lang]
    question=query(lang,name,term[target])
    prompt={'pt':'Premissas:','en':'Premises:','es':'Premisas:'}[lang]
    answer={'pt':'Resposta:','en':'Answer:','es':'Respuesta:'}[lang]
    text=f'{prompt} '+ ' '.join(premises)+f'\n{question}\n{answer} '+(yes if result else no)
    return {'text':text,'lang':lang,'domain':'logic','template':template,
            'label_entails':result,'premises':premises,'question':question,
            'answer':yes if result else no,'source_id':'maia-synthetic-logic-v1',
            'license':'Project-generated; verify publication policy',
            'attribution':'Maia Lite corpus generator', 'origin_url':'local://maia-logic-v1'}

def generate_logic(split,per_lang):
    templates=TEMPLATES_TRAIN if split=='train' else TEMPLATES_EVAL
    result=[]
    for lang in LANGS:
        rng=random.Random(SEED+{'pt':0,'en':100000,'es':200000}[lang]+(0 if split=='train' else 500000))
        seen=set(); attempts=0
        while sum(r['lang']==lang for r in result)<per_lang:
            attempts+=1
            if attempts>per_lang*100: raise RuntimeError('Too many duplicate logical examples')
            template=templates[(attempts-1)%len(templates)]
            row=make_logic(lang,template,rng)
            fp=fingerprint(row['text'])
            if fp in seen: continue
            seen.add(fp); row['split']=split; result.append(row)
    return result

# Basic soundness tests for the finite-model checker
assert verify_entailment([('all',0,1)],[(0,True)],1,2)
assert not verify_entailment([('all',0,1)],[(1,True)],0,2)
assert not verify_entailment([('all',0,1)],[(0,False)],1,2)
assert not verify_entailment([('none',0,1)],[(0,True)],1,2)
print('Logical verifier checks passed.')

## 4. Assemble corpus, enforce duplicate controls, and isolate evaluation

The evaluation split is **never** placed in `source`. Exact normalized duplicate removal is applied across authentic texts and generated examples. Cross-lingual paraphrase duplicates and semantic near-duplicates are **not detected**; these require a separate audit. In production, reserve entire document collections, authors or sources for evaluation, rather than relying only on text hashes.

**Safety:** this notebook refuses to overwrite existing corpus outputs unless you explicitly opt in. It writes atomic files (`.tmp` then replace) and a manifest with checksums.

In [ ]:
GENERATE_SYNTHETIC_LOGIC=True
ALLOW_LOGIC_ONLY_PILOT=True  # Allows pipeline smoke test, NOT production approval
OVERWRITE_GENERATED=False

registered=list(iter_registered())
print('Registered authentic records:',len(registered))
logic_train=generate_logic('train',LOGIC_COUNTS_TRAIN_PER_LANG) if GENERATE_SYNTHETIC_LOGIC else []
logic_eval=generate_logic('eval',LOGIC_COUNTS_EVAL_PER_LANG) if GENERATE_SYNTHETIC_LOGIC else []

if not registered and not ALLOW_LOGIC_ONLY_PILOT:
    raise RuntimeError('No authentic licensed texts. Add to inbox and registry before building corpus.')

seen=set(); rejected=collections.Counter()
def dedup(rows):
    out=[]
    for rec in rows:
        key=fingerprint(rec['text'])
        if key in seen:
            rejected['duplicate']+=1; continue
        seen.add(key)
        out.append(rec)
    return out

# Holdout first: if an accidental exact duplicate appears in train, holdout wins.
logic_eval=dedup(logic_eval)
registered=dedup(registered)
logic_train=dedup(logic_train)
train_rows=registered+logic_train
assert all(r['lang'] in LANGS and r['domain'] in DOMAINS for r in train_rows)
assert set(r['template'] for r in logic_train).isdisjoint(set(r['template'] for r in logic_eval))
print('Accepted train:',len(train_rows),'held-out logic:',len(logic_eval),'rejected:',dict(rejected))

# Keep one file per lang/domain; names begin with 'maia_corpus_' for safe identification.
expected=[SOURCE/lang/domain/'maia_corpus_v030.jsonl' for lang in LANGS for domain in DOMAINS]
expected += [EVAL/'maia_logic_holdout_v030.jsonl',REPORTS/'maia_corpus_v030_manifest.json']
if not OVERWRITE_GENERATED and any(p.exists() for p in expected):
    raise FileExistsError('Existing generated outputs detected. Review them before setting OVERWRITE_GENERATED=True.')

def write_jsonl_atomic(path,rows):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    with tmp.open('w',encoding='utf-8') as f:
        for row in rows: f.write(json.dumps(row,ensure_ascii=False)+'\n')
    os.replace(tmp,path)

for lang in LANGS:
    for domain in DOMAINS:
        group=[r for r in train_rows if r['lang']==lang and r['domain']==domain]
        if group: write_jsonl_atomic(SOURCE/lang/domain/'maia_corpus_v030.jsonl',group)
write_jsonl_atomic(EVAL/'maia_logic_holdout_v030.jsonl',logic_eval)
print('Written corpus files. Do not start full pretraining before the audit below.')

## 5. Composition audit, license inventory, token-proxy estimates

In [ ]:
def audit_rows(rows):
    df=pd.DataFrame([{'lang':r['lang'],'domain':r['domain'],'chars':len(r['text']),
                      'words':len(r['text'].split()),'source_id':r['source_id'],
                      'license':r['license']} for r in rows])
    if df.empty: return df,{},{}
    by_lang=df.groupby('lang')['chars'].sum().to_dict()
    by_domain=df.groupby('domain')['chars'].sum().to_dict()
    total=sum(by_lang.values())
    return df,{k:round(by_lang.get(k,0)/total,4) for k in LANGS}, {k:round(by_domain.get(k,0)/total,4) for k in DOMAINS}

df,lang_share,domain_share=audit_rows(train_rows)
print('Character shares (proxy ONLY; measure actual tokens after tokenizer training):')
print('Languages:',lang_share,'target:',TARGET_LANG)
print('Domains:',domain_share,'target:',TARGET_DOMAIN)
if len(df):
    display(df.groupby(['lang','domain']).agg(documents=('chars','size'),chars=('chars','sum'),words=('words','sum')))
    display(df.groupby(['source_id','license']).size().rename('documents').reset_index())

missing=[(lang,domain) for lang in LANGS for domain in DOMAINS
         if not any(r['lang']==lang and r['domain']==domain for r in train_rows)]
print('Missing language/domain cells:',missing)
if not registered:
    print('WARNING: LOGIC-ONLY PILOT. NOT SUITABLE FOR FULL MODEL PRETRAINING.')
if any(abs(lang_share.get(k,0)-v)>.10 for k,v in TARGET_LANG.items()):
    print('WARNING: Language mix differs materially from the research target.')
if any(abs(domain_share.get(k,0)-v)>.10 for k,v in TARGET_DOMAIN.items()):
    print('WARNING: Domain mix differs materially from the research target.')

## 6. Integrity manifest and readiness gate

The manifest captures file hashes and audit statistics. **A successful run does not certify the corpus for publication or production training.** Synthetic logic can dominate a pilot unless authentic data are supplied. The `source` directory may contain other files; the downstream pretraining notebook will read **all** `.jsonl` and `.txt` files recursively, so inspect it before launching training.

In [ ]:
def sha256_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

output_files=sorted(p for p in SOURCE.rglob('maia_corpus_v030.jsonl'))+[EVAL/'maia_logic_holdout_v030.jsonl']
manifest={'version':'0.3.0','seed':SEED,'generated_utc':pd.Timestamp.utcnow().isoformat(),
          'documents_train':len(train_rows),'documents_logic_eval':len(logic_eval),
          'authentic_documents':len(registered),'synthetic_logic_documents':len(logic_train),
          'language_share_by_chars':lang_share,'domain_share_by_chars':domain_share,
          'target_language':TARGET_LANG,'target_domain':TARGET_DOMAIN,
          'duplicates_rejected':dict(rejected),'missing_cells':missing,
          'registered_sources':SOURCE_REGISTRY,
          'files':{str(p.relative_to(ROOT)):sha256_file(p) for p in output_files if p.exists()},
          'production_ready':bool(registered and not missing and
             all(abs(lang_share.get(k,0)-v)<=.10 for k,v in TARGET_LANG.items()) and
             all(abs(domain_share.get(k,0)-v)<=.10 for k,v in TARGET_DOMAIN.items()))}
mp=REPORTS/'maia_corpus_v030_manifest.json'
mp.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Manifest:',mp)
print('PRODUCTION READINESS (preliminary, not license or quality certification):',manifest['production_ready'])
print('Logic examples:')
for row in (logic_train[:1]+logic_eval[:1]): print('\n',row['template'],row['lang'],'\n',row['text'])

## 7. Optional: produce a balanced **sampled** training subset

The main `source` output preserves every approved authentic record and every generated training example. The target shares are **not enforced**. If you need a pilot with a controlled mix, build a separate directory, inspect counts, and point the pretraining notebook's `SOURCE` variable to that directory. **Sampling with replacement creates repeated examples, not new knowledge**; this cell samples *without replacement*, and warns when targets cannot be achieved. Sampling here uses document counts, not tokenizer tokens, so final shares must be audited downstream.

In [ ]:
MAKE_BALANCED_PILOT=False
BALANCED_DIR=ROOT/'corpus'/'balanced_pilot_source'
BALANCED_TOTAL_DOCS=5000
if MAKE_BALANCED_PILOT:
    rng=random.Random(SEED)
    by_group=collections.defaultdict(list)
    for r in train_rows: by_group[(r['lang'],r['domain'])].append(r)
    selected=[]; shortages=[]
    for lang in LANGS:
        for domain in DOMAINS:
            desired=round(BALANCED_TOTAL_DOCS*TARGET_LANG[lang]*TARGET_DOMAIN[domain])
            pool=by_group[(lang,domain)]
            if len(pool)<desired: shortages.append((lang,domain,desired,len(pool)))
            selected+=rng.sample(pool,min(desired,len(pool)))
    if shortages:
        print('Insufficient unique examples (no oversampling performed):',shortages)
    if BALANCED_DIR.exists() and any(BALANCED_DIR.rglob('*.jsonl')):
        raise FileExistsError(f'Balanced directory already populated: {BALANCED_DIR}')
    for lang in LANGS:
        for domain in DOMAINS:
            group=[r for r in selected if r['lang']==lang and r['domain']==domain]
            if group: write_jsonl_atomic(BALANCED_DIR/lang/domain/'balanced_v030.jsonl',group)
    print('Selected',len(selected),'documents at',BALANCED_DIR)
else:
    print('Balanced pilot disabled. Main source remains unchanged.')

## 8. Required steps before a full 355M pretraining run

1. Acquire real PT/EN/ES material with documented usage rights; register each source, preserve document boundaries and provenance.
2. Add **authentic dialogues**, explanations, essays, creative writing and varied open-ended instructions. Logic-only synthetic text is insufficient for a Chat model.
3. Inspect PII, copyright, language detection, boilerplate, toxicity and semantic near-duplicates; use source-level held-out splits.
4. Extend the logic generator with **more structural families**, negative conclusions, quantified relations, nested conditionals, ambiguous/insufficient premises and non-entailment. Verify labels with a formal checker appropriate to each fragment. The current generator covers only a deliberately narrow fragment.
5. Train and compare 32k/40k/50k byte-level BPE tokenizers **on training data only**; evaluate token-per-character by language and domain.
6. Measure real token shares and sampling efficiency; freeze a corpus version with checksums. The downstream pretraining notebook currently performs its own hash-based validation split from `source`, which is **not** a source-level split. Improve it before scientific evaluation.
7. Preserve historical regression probes, especially **“Fale sobre a saudade.”** versus **“O que é a saudade?”**, with corresponding EN/ES variants.
8. Run the small pilot and compare a matched general-corpus control before authorizing the full training budget.